# 연관 과목 이유 문장 엔진 실험 v2

v1 결과: LLM(M4)은 자연스럽지만 27%가 근거에 없는 내용을 지어냈고, 단어 비교 검사(M5)는 정상 문장까지 83%를 대체했다.
근거로 "Introduction", "Course overview" 같은 문장이 뽑히면 어떤 방법도 쓸모가 없었다.

v2는 파인튜닝 없이 **파이프라인**을 고친다. 연구실·쌍은 v1과 같다(12곳 × 상위 5개).

| 변경 | 내용 |
|---|---|
| ① 근거 품질 기준 | 짧거나 일반적인 문장(소개·개요·장 번호·"이 과목은…")을 근거 후보에서 제외. 남은 최고 근거도 약하면 **이유 생략** |
| ② 근거 번역 | 영어 근거를 LLM으로 한국어 한 줄로 옮겨 인용(M2k)과 검사에 사용 |
| ③ LLM 개선 | 금지 예시를 넣은 프롬프트 + 깨진 글자(한자·`_`·문장 중간 영어 조각) 정리 |
| ④ 새 검사 | 단어 비교 대신 재순위 모델로 "생성 문장 ↔ 근거(원문+번역)" 일치도 측정. 기준점은 M2k(정확한 인용) 점수 분포로 자동 결정 |

| 방법 | 설명 |
|---|---|
| M2 | v1 근거 인용 (영어 근거 그대로) |
| M2k | 한국어로 옮긴 근거 인용 |
| M4b | 개선된 LLM 생성 |
| M5b | M4b + v1 검사(단어 비교), 실패 시 M2k |
| M6 | M4b + 새 검사(재순위 일치도), 실패 시 M2k |

### 준비
- **런타임 → 런타임 유형 변경 → T4 GPU**
- 업로드 (서버 저장소 `data/`): `courses.jsonl`, `lab_profiles.jsonl`, `course_profiles.jsonl`, `concepts.json`
- 업로드 (서버 `experiments/bge_m3/V4/`): `rankings_v4.json`

In [ ]:
# 1. GPU 확인
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('⚠ GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 선택하고 다시 실행하세요.')

In [ ]:
# 2. 라이브러리 설치
!pip install -q sentence-transformers transformers accelerate bitsandbytes

In [ ]:
# 3. 데이터 업로드
import os
from google.colab import files
need = ['courses.jsonl', 'lab_profiles.jsonl', 'course_profiles.jsonl', 'concepts.json', 'rankings_v4.json']
missing = [f for f in need if not os.path.exists(f)]
if missing:
    print('업로드할 파일:', missing)
    files.upload()
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f'아직 없는 파일: {missing}'
print('데이터 준비 완료')

In [ ]:
# 4. 데이터와 평가 대상 쌍
import json, re, time, random, statistics
from collections import defaultdict, Counter
import numpy as np

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

courses = load_jsonl('courses.jsonl')
course_by_id = {c['record_id']: c for c in courses}
profiles = [p for p in load_jsonl('lab_profiles.jsonl') if p.get('status') != 'insufficient']
prof_by_id = {p['lab_id']: p for p in profiles}
course_profiles = {c['record_id']: c for c in load_jsonl('course_profiles.jsonl')}
concepts = {c['id']: c for c in json.load(open('concepts.json', encoding='utf-8'))['concepts']}
rankings = json.load(open('rankings_v4.json', encoding='utf-8'))['R2']

N_LABS, TOP_K, SEED = 12, 5, 7          # 연구실 수 × 상위 과목 수 = 평가 쌍 수
labs = random.Random(SEED).sample([p['lab_id'] for p in profiles], 20)[:N_LABS]   # v2~v4 채점 연구실과 같은 순서
pairs = [(lid, c['record_id']) for lid in labs for c in rankings[lid]['top30'][:TOP_K]]
print(f'평가 쌍 {len(pairs)}개 (연구실 {len(labs)}곳 × 상위 {TOP_K}개)')

def course_name(c):
    return c.get('course_name_kr') or c.get('course_name_en') or c['course_code']

## 1단계: 근거 선택 + 품질 기준

- 후보 문장에서 시험·과제·소개·개요·장 번호·"이 과목은/학생들은" 같은 일반 문장을 뺀다.
- 재순위 점수로 가장 잘 맞는 (주제 문장, 과목 문장)을 고른다.
- 남은 최고 근거의 점수가 전체 쌍의 하위 `GATE_PCT` 백분위보다 낮거나 내용어가 2개 미만이면 **이유 생략**으로 표시한다.

In [ ]:
from sentence_transformers import CrossEncoder
device = 'cuda' if torch.cuda.is_available() else 'cpu'
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)
if device == 'cuda':
    reranker.model.half()
def rr_scores(pairs_, bs=64):
    raw = np.asarray(reranker.predict(pairs_, batch_size=bs, show_progress_bar=len(pairs_) > 500), dtype=np.float32)
    return raw if (raw.min() >= 0 and raw.max() <= 1) else 1 / (1 + np.exp(-raw))

SENT_SPLIT = re.compile(r'(?<=[.!?。])\s+|(?<=다\.)\s*|\s*[•■▪\-–]\s+|\s*/\s+')
JUNK = re.compile(r'(see attached|attached file|첨부|tbd|tba|mid-?term|final|exam|quiz|homework|assignment|grading|attendance|presentation|review|holiday|no class|assumed|familiar with|prerequisite|중간|기말|시험|퀴즈|과제|출석|성적|평가|발표|휴강|공휴일|선수과목)', re.I)
GENERIC = re.compile(r'^(introduction|course (overview|introduction)|overview|class introduction.*|summary|conclusion|basics?|chapter.*|\(?chapter.*|ch\.?\s*\d.*|part [ivx\d]+.*|the (course|objective|goal) (material|of this (class|course)).*|(this|the) course (will|is|aims|covers|provides|introduces|deals).*|this class .*|in this (course|class).*|students will.*|it is hoped.*|본 (과목|강의|교과목)(은|에서는|을).*|이 (과목|강의)(은|에서는).*|학생들은.*)$', re.I)
CONTENT = re.compile(r'[A-Z]{2,}|[A-Za-z]{4,}|[가-힣]{2,}')
STOPW = {'this', 'that', 'with', 'from', 'will', 'course', 'class', 'students', 'student', 'learn', 'introduction', 'overview', 'basic', 'basics',
         '과목', '강의', '학습', '이해', '소개', '기본', '기초', '내용', '학생'}

def informative(s):
    if JUNK.search(s) or GENERIC.match(s.strip()):
        return False
    words = [w for w in CONTENT.findall(s) if w.lower() not in STOPW]
    return len(words) >= 2 or (len(words) == 1 and len(words[0]) >= 8)

def course_sentences(c):
    out = []
    for where, text in [('과목 설명', c.get('description') or ''), ('학습목표', c.get('objectives') or '')]:
        for s in SENT_SPLIT.split(text):
            s = re.sub(r'\s+', ' ', s).strip()
            if 15 <= len(s) <= 300 and informative(s):
                out.append((where, s))
    for line in re.split(r'\n+', str(c.get('course_plan') or '')):
        line = re.sub(r'^[\W\d_]*(week|주차|lecture|chapter)?\s*[\d.]*\s*[:.)-]?\s*', '', line.strip(), flags=re.I)
        line = re.sub(r'\s+', ' ', line)
        if 12 <= len(line) <= 200 and informative(line):
            out.append(('강의계획', line))
    return list(dict.fromkeys(out))[:60]

def primary_topics(lid):
    return [t for t in prof_by_id[lid]['topics'] if t.get('weight', 'primary') == 'primary']

t0 = time.time()
batch, owners = [], []
for pi, (lid, rid) in enumerate(pairs):
    sents = course_sentences(course_by_id[rid])
    for ti, t in enumerate(primary_topics(lid)):
        for si, (where, s) in enumerate(sents):
            batch.append((f"{t['label_ko']}: {t['quote']}", s))
            owners.append((pi, ti, si))
scores = rr_scores(batch)
best = {}
for (pi, ti, si), sc in zip(owners, scores):
    if pi not in best or sc > best[pi][0]:
        best[pi] = (float(sc), ti, si)

GATE_PCT = 15   # 근거 점수 하위 15%는 이유 생략
gate = float(np.percentile([b[0] for b in best.values()], GATE_PCT)) if best else 1.0
evidence = []
for pi, (lid, rid) in enumerate(pairs):
    c = course_by_id[rid]; sents = course_sentences(c)
    e = {'lab_id': lid, 'record_id': rid, 'lab': prof_by_id[lid]['professors'][0].split(' (')[0],
         'lab_summary': prof_by_id[lid]['summary_ko'], 'course': course_name(c), 'course_en': c.get('course_name_en') or '',
         'code': c['course_code']}
    if pi in best and best[pi][0] >= gate:
        sc, ti, si = best[pi]; t = primary_topics(lid)[ti]; where, s = sents[si]
        e.update({'topic': t['label_ko'], 'topic_quote': t['quote'], 'where': where, 'sentence': s, 'rr': round(sc, 4), 'skip': False})
    else:
        t = primary_topics(lid)[0]
        e.update({'topic': t['label_ko'], 'topic_quote': t['quote'], 'where': '', 'sentence': '', 'rr': round(best.get(pi, (0.0,))[0], 4), 'skip': True})
    evidence.append(e)
print(f'근거 선택 {len(batch)}쌍 {time.time()-t0:.1f}s, 생략 기준 점수 {gate:.3f}')
print(f"이유 생략 {sum(e['skip'] for e in evidence)} / {len(evidence)}쌍")
for e in evidence[:5]:
    print(f"  [{e['lab']}] {e['topic']} ↔ {e['course']}: {'(생략)' if e['skip'] else e['where'] + ' — ' + e['sentence'][:80]}")

## 2단계: LLM 로드, 근거 번역(M2k), 개선된 생성(M4b)

기본 모델은 `Qwen/Qwen2.5-7B-Instruct`(4비트). 한국어 비교는 `LLM_ID`를 `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct`(연구용 라이선스)로 바꿔 다시 실행한다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
LLM_ID = 'Qwen/Qwen2.5-7B-Instruct'
# 이 셀을 다시 실행할 때 이전에 올린 LLM이 GPU에 남지 않도록 정리
import gc
for _name in ['llm', 'tok']:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU 여유 메모리 {free/1e9:.1f} / {total/1e9:.1f} GB (7B 4비트에 약 6GB 필요)')
    if free < 6.5e9:
        print('⚠ 메모리가 부족합니다. 런타임 → 세션 다시 시작 후 처음부터 실행하세요.')
t0 = time.time()
tok = AutoTokenizer.from_pretrained(LLM_ID, trust_remote_code=True)
tok.padding_side = 'left'
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID, device_map={'': 0} if torch.cuda.is_available() else 'cpu', trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'))
print(f'LLM 로드 {time.time()-t0:.1f}s')

def generate(messages_list, max_new=120, bs=8):
    outs = []
    for i in range(0, len(messages_list), bs):
        texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_list[i:i+bs]]
        enc_in = tok(texts, return_tensors='pt', padding=True).to(llm.device)
        with torch.no_grad():
            out = llm.generate(**enc_in, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        for ids in out[:, enc_in['input_ids'].shape[1]:]:
            outs.append(tok.decode(ids, skip_special_tokens=True).strip())
        print(f'  {min(i+bs, len(messages_list))}/{len(messages_list)}')
    return outs

HAN = re.compile(r'[\u4e00-\u9fff\u3040-\u30ff]')
def clean(text):
    t = text.strip().split(chr(10))[0].strip().strip('"').strip()
    t = HAN.sub('', t)
    t = re.sub(r'_[A-Za-z]+', '', t)                       # 제어_technologies → 제어
    t = re.sub(r'\s{2,}', ' ', t).replace('。', '.').strip()
    return t

# 근거 번역 (영어가 섞인 근거만)
active = [e for e in evidence if not e['skip']]
need_tr = [e for e in active if re.search(r'[A-Za-z]{3,}', e['sentence'])]
TR_SYS = '다음 대학 강의 자료 문장을 자연스러운 한국어 명사구 또는 한 문장으로 옮겨라. 전공 용어는 널리 쓰는 한국어 용어를 쓰고 괄호에 영어를 붙여도 된다. 내용을 더하거나 빼지 마라. 번역만 출력하라.'
t0 = time.time()
trs = generate([[{'role': 'system', 'content': TR_SYS}, {'role': 'user', 'content': e['sentence']}] for e in need_tr], max_new=80)
for e, t in zip(need_tr, trs):
    e['sentence_ko'] = clean(t) or e['sentence']
for e in active:
    e.setdefault('sentence_ko', e['sentence'])
print(f'번역 {len(need_tr)}건 {time.time()-t0:.1f}s. 예: {need_tr[0]["sentence"][:60] if need_tr else ""} → {need_tr[0]["sentence_ko"] if need_tr else ""}')

SYSTEM = ('너는 대학생에게 연구실과 수업의 연결을 설명하는 도우미다. [근거]에 적힌 내용만 사용해 한국어 해요체 한 문장(40~80자)으로 '
          '이 수업에서 배우는 것이 이 연구실 주제와 어떻게 이어지는지 설명하라.\n'
          '규칙: 근거에 없는 사실·용어·연구실 프로젝트·실험을 만들지 마라. "직접", "반드시", "필수", "최고" 같은 단정 표현을 쓰지 마라. '
          '영어 단어를 한국어 문장 중간에 섞지 말고 한국어 용어를 써라. 문장만 출력하라.\n'
          '나쁜 예: "이 수업에서 배운 인공지능 연산은 포논 시뮬레이션 연구실 프로젝트에 직접 적용돼요." (근거에 없는 연결, 프로젝트 언급, 단정)\n'
          '좋은 예: "강의계획의 PID 제어를 배우면 전력 변환기 제어 연구의 기본 개념을 이해하는 데 도움이 돼요."')
def m4_messages(e):
    user = (f"[근거]\n연구실 주제: {e['topic']}\n연구실 자료 문장: {e['topic_quote']}\n"
            f"수업: {e['course']}\n수업 자료({e['where']}): {e['sentence_ko']}")
    return [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}]
t0 = time.time()
gens = generate([m4_messages(e) for e in active])
for e, g_ in zip(active, gens):
    e['M4b'] = clean(g_)
print(f'M4b 생성 {time.time()-t0:.1f}s. 예: {active[0]["M4b"] if active else ""}')

## 3단계: 인용 문장과 두 가지 검사

- **M2 / M2k**: 근거 인용 (원문 / 한국어)
- **v1 검사(M5b)**: 생성 문장의 내용어가 근거(원문+번역)에 없는 비율이 0.35 초과면 실패
- **새 검사(M6)**: 재순위 모델로 `(근거 원문 + 번역, 생성 문장)` 일치도를 잰다. 기준점은 **M2k 문장(정확한 인용)의 일치도 하위 `FAITH_PCT` 백분위**로 자동 설정한다. 과장·깨짐이 있으면 무조건 실패.

In [ ]:
def short(s, n=70):
    s = re.sub(r'\s+', ' ', s).strip()
    return s if len(s) <= n else s[:n].rsplit(' ', 1)[0] + '…'
for e in active:
    e['M2'] = f"이 연구실의 '{e['topic']}' 연구와 연결돼요. 이 과목의 {e['where']}에서 \"{short(e['sentence'])}\"를 다뤄요."
    e['M2k'] = f"이 연구실의 '{e['topic']}' 연구와 연결돼요. 이 과목의 {e['where']}에서 '{short(e['sentence_ko'])}'을(를) 다뤄요."

FRAME = {'연구', '연구실', '과목', '수업', '배우', '배워', '배울', '배우면', '다뤄', '다루', '연결', '도움', '필요', '관련', '지식', '내용',
         '이해', '학습', '활용', '기초', '기본', '바탕', '주제', '분야', '통해', '대한', '위한', '있어요', '있습니다', '있는', '하는', '돼요', '이에요', '에서', '개념'}
OVERCLAIM = re.compile(r'(반드시|필수|최고|완벽|보장|무조건|직접적으로|직접 적용)')
GARBLE_RE = re.compile(r'([\u4e00-\u9fff]|_[A-Za-z]|[A-Za-z]{3,}[가-힣]|[가-힣][A-Za-z]{3,}|\b[a-z]{3,}\b)')
class _Garble:   # 괄호 안 영어 병기는 허용하고, 그 밖의 소문자 영어 조각·한글에 붙은 영어·한자는 깨짐으로 본다
    def search(self, t):
        return GARBLE_RE.search(re.sub(r'\([^)]*\)', '', t))
GARBLE = _Garble()
def content_words(s):
    return [w for w in re.findall(r'[A-Za-z]{3,}|[가-힣]{2,}', s) if w not in FRAME and not any(w.startswith(f) for f in FRAME)]
def unsupported_ratio(text, e):
    src = ' '.join([e['topic'], e['topic_quote'], e['course'], e['course_en'], e['sentence'], e['sentence_ko']]).lower()
    words = content_words(text)
    if not words:
        return 0.0
    return sum(w.lower() not in src and w[:2].lower() not in src for w in words) / len(words)

def faith(texts):
    return rr_scores([(f"{e['topic']}: {e['topic_quote']} / {e['course']}: {e['sentence']} ({e['sentence_ko']})", t) for e, t in zip(active, texts)])

f_m2k = faith([e['M2k'] for e in active])
f_m4b = faith([e['M4b'] for e in active])
FAITH_PCT = 10
FAITH_MIN = float(np.percentile(f_m2k, FAITH_PCT))
for e, f2, f4 in zip(active, f_m2k, f_m4b):
    e['faith_M2k'], e['faith_M4b'] = round(float(f2), 4), round(float(f4), 4)
    hard_bad = bool(OVERCLAIM.search(e['M4b']) or GARBLE.search(e['M4b']) or len(e['M4b']) < 15)
    e['M5b_fallback'] = bool(hard_bad or unsupported_ratio(e['M4b'], e) > 0.35)
    e['M6_fallback'] = bool(hard_bad or f4 < FAITH_MIN)
    e['M5b'] = e['M2k'] if e['M5b_fallback'] else e['M4b']
    e['M6'] = e['M2k'] if e['M6_fallback'] else e['M4b']
print(f'새 검사 기준점 FAITH_MIN = {FAITH_MIN:.3f} (M2k 일치도 하위 {FAITH_PCT}%)')
print(f"일치도 중앙값: M2k {np.median(f_m2k):.3f}, M4b {np.median(f_m4b):.3f}")
print(f"대체율: M5b {np.mean([e['M5b_fallback'] for e in active]):.0%}, M6 {np.mean([e['M6_fallback'] for e in active]):.0%}")

METHODS = ['M2', 'M2k', 'M4b', 'M5b', 'M6']
import pandas as pd
auto = {}
for m in METHODS:
    texts = [e[m] for e in active]
    auto[m] = {'평균 길이(자)': round(statistics.mean(len(t) for t in texts), 1),
               '근거 밖 단어 비율': round(statistics.mean(unsupported_ratio(t, e) for t, e in zip(texts, active)), 3),
               '과장 표현(건)': sum(bool(OVERCLAIM.search(t)) for t in texts),
               '깨진 글자(건)': sum(bool(GARBLE.search(t)) for t in texts),
               '서로 다른 문장 비율': round(len(set(texts)) / len(texts), 3)}
print(f"이유 생략 쌍: {sum(e['skip'] for e in evidence)} (평가 대상 {len(active)}쌍)")
pd.DataFrame(auto)

In [ ]:
# 나란히 보기
for e in active[:8]:
    print(f"\n■ {e['lab']} · {e['topic']} ↔ {e['code']} {e['course']}  (근거 {e['rr']}, 일치도 M4b {e['faith_M4b']})")
    print(f"   근거: {e['where']} — {short(e['sentence'], 80)}")
    print(f"   번역: {e['sentence_ko']}")
    for m in METHODS:
        print(f'   {m}: {e[m]}')
print('\n이유 생략된 쌍 예시:')
for e in [x for x in evidence if x['skip']][:5]:
    print(f"   {e['lab']} ↔ {e['code']} {e['course']} (최고 근거 점수 {e['rr']})")

## 4단계: Claude Code 채점용 파일 (블라인드)

v1과 같은 기준: **faithful** 0/1, **specific**·**fluent**·**useful** 1~3. 이유 생략된 쌍은 채점에서 빠지고, 생략이 적절했는지 따로 0/1로 채점한다(`skip_ok`).

In [ ]:
packet, key = [], {}
for pi, e in enumerate(evidence):
    base = {'pair': f'p{pi:03d}', 'lab': e['lab'], 'lab_summary': e['lab_summary'], 'topic': e['topic'],
            'topic_quote': e['topic_quote'][:250], 'course': f"{e['code']} {e['course']} / {e['course_en']}",
            'course_description': (course_by_id[e['record_id']].get('description') or '')[:300]}
    if e['skip']:
        packet.append({**base, 'skipped': True, 'reasons': [{'id': f'p{pi:03d}|skip', 'text': '(근거가 약해 이유 문장을 생략함)'}]})
        key[f'p{pi:03d}|skip'] = ['SKIP']
        continue
    texts = {}
    for m in METHODS:
        texts.setdefault(e[m], []).append(m)
    items = list(texts.items())
    random.Random(pi).shuffle(items)
    reasons = []
    for j, (t, ms) in enumerate(items):
        rid = f'p{pi:03d}|r{j+1}'
        reasons.append({'id': rid, 'text': t}); key[rid] = ms
    packet.append({**base, 'skipped': False, 'evidence_sentence': e['sentence'], 'reasons': reasons})
json.dump({'instructions': '각 reason에 faithful(0/1), specific(1~3), fluent(1~3), useful(1~3). '
                           'skipped 쌍의 "|skip" 항목에는 skip_ok(0/1: 생략이 적절하면 1). '
                           '반환 형식: {"<id>": {"faithful":..,"specific":..,"fluent":..,"useful":..}} 또는 {"<id>|skip": {"skip_ok":0|1}}',
           'pairs': packet}, open('reason_packet.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
json.dump(key, open('reason_key.json', 'w', encoding='utf-8'), ensure_ascii=False)
print(f'채점용 파일: 쌍 {len(packet)}개(생략 {sum(p["skipped"] for p in packet)}), 항목 {len(key)}개 → reason_packet.json')

## 결과 저장·다운로드

In [ ]:
import zipfile
json.dump(evidence, open('reasons_v2.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
pd.DataFrame(auto).to_csv('reason_auto_metrics_v2.csv', encoding='utf-8-sig')
json.dump({'N_LABS': N_LABS, 'TOP_K': TOP_K, 'SEED': SEED, 'LLM_ID': LLM_ID, 'GATE_PCT': GATE_PCT, 'GATE_SCORE': gate,
           'FAITH_PCT': FAITH_PCT, 'FAITH_MIN': FAITH_MIN, 'RERANKER': 'BAAI/bge-reranker-v2-m3'},
          open('reason_settings_v2.json', 'w'), indent=1)
with zipfile.ZipFile('reason_results_v2.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['reasons_v2.json', 'reason_auto_metrics_v2.csv', 'reason_settings_v2.json', 'reason_packet.json', 'reason_key.json']:
        z.write(f)
print('저장 완료: reason_results_v2.zip')
files.download('reason_results_v2.zip')